In [ ]:
import numpy as np
import os
import torch
from transformers import BertTokenizer, BertModel

DATA_PATH = '../../data'
DB_FILE   = 'FairCVdb.npy'

fairCV = np.load(os.path.join(DATA_PATH, DB_FILE), allow_pickle=True).item()
bios_train_original = fairCV['Bios Train'][:, 0]
bios_test_original  = fairCV['Bios Test'][:, 0]
bios_train_blinded  = fairCV['Bios Train'][:, 1]
bios_test_blinded   = fairCV['Bios Test'][:, 1]

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model.to(device)

In [ ]:
# Create embeddings for bios

def create_embeddings(bios, batch_size):
    embeddings = []

    for i in range(0, len(bios), batch_size):
        bio = bios[i:i + batch_size].tolist()

        # returns a dictionary
        encoded = tokenizer(bio,
                            padding=True, # adds padding to match longest sequence
                            truncation=True,
                            max_length=256,
                            return_tensors='pt') # pt = PyTorch tensors

        encoded = {key: value.to(device) for key, value in encoded.items()} # move each tensor to GPU

        with torch.no_grad(): # turn off gradient tracking, takes time and memory
            # run it through the model
            output = model(input_ids=encoded['input_ids'], attention_mask=encoded['attention_mask'])

            # append the embedding to your list
            embeddings.append(output.last_hidden_state[:, 0, :].cpu().numpy())

    return np.vstack(embeddings)


# ORIGINAL BIOS
if not os.path.exists(os.path.join(DATA_PATH, 'original_bios_train_bert.npy')):
    text_train_original = create_embeddings(bios_train_original, 128)
    np.save(os.path.join(DATA_PATH, 'original_bios_train_bert.npy'), text_train_original)
else:
    text_train_original = np.load(os.path.join(DATA_PATH, 'original_bios_train_bert.npy'))

if not os.path.exists(os.path.join(DATA_PATH, 'original_bios_test_bert.npy')):
    text_test_original = create_embeddings(bios_test_original, 128)
    np.save(os.path.join(DATA_PATH, 'original_bios_test_bert.npy'), text_test_original)
else:
    text_test_original = np.load(os.path.join(DATA_PATH, 'original_bios_test_bert.npy'))

# BLINDED BIOS
if not os.path.exists(os.path.join(DATA_PATH, 'blinded_bios_train_bert.npy')):
    text_train_blinded = create_embeddings(bios_train_blinded, 128)
    np.save(os.path.join(DATA_PATH, 'blinded_bios_train_bert.npy'), text_train_blinded)
else:
    text_train_blinded = np.load(os.path.join(DATA_PATH, 'blinded_bios_train_bert.npy'))

if not os.path.exists(os.path.join(DATA_PATH, 'blinded_bios_test_bert.npy')):
    text_test_blinded = create_embeddings(bios_test_blinded, 128)
    np.save(os.path.join(DATA_PATH, 'blinded_bios_test_bert.npy'), text_test_blinded)
else:
    text_test_blinded = np.load(os.path.join(DATA_PATH, 'blinded_bios_test_bert.npy'))


print(f"Original train: {text_train_original.shape}")
print(f"Original test: {text_test_original.shape}")
print(f"Blinded train: {text_train_blinded.shape}")
print(f"Blinded test: {text_test_blinded.shape}")
print("Train Identical:", np.array_equal(text_train_original, text_train_blinded))
print("Test Identical:", np.array_equal(text_test_original, text_test_blinded))

The tokenizer converts raw strings into token IDs. It returns a dictionary of pytorch
tensors with two keys (ignoring token_type_ids):
- **input_ids**: the integer IDs for each token according to BERTs vocabulary
- **attention mask**: shorter bios get padded with zeros to match the longest one, the attention mask tells BERT which positions are real tokens (1) and which are just padding (0).